In [1]:
import numpy as np, json, pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers, optimizers, callbacks
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import classification_report, f1_score

In [2]:
df = pd.read_csv("artirilmis_temizlenmis.csv", encoding="utf-8-sig")
print(df.head())

                                            yorumlar    duygu
0                 Fiyatına göre gayet güzel bir ürün  pozitif
1  sıfır kutusunda sorunsuz fiyatı ram 3gb olduğu...  pozitif
2  makinanın heryeri yapışkanlı gibi tuttuğum yer...     notr
3                şahane telefon satıcıya selam olsun  pozitif
4               kalitesiz bir ürün tavsiye etmiyorum  negatif


In [3]:
data = np.load("preprocessed_arrays.npz")
X_train_pad, y_train_cat = data["X_train_pad"], data["y_train_cat"]
X_val_pad,   y_val_cat   = data["X_val_pad"],   data["y_val_cat"]

with open("label_encoder_classes.json","r",encoding="utf-8") as f:
    classes = json.load(f)

In [4]:
with open("tokenizer.json","r",encoding="utf-8") as f:
    tok = tokenizer_from_json(f.read())

In [5]:
MAX_LEN   = X_train_pad.shape[1]
MAX_VOCAB = tok.num_words or (len(tok.word_index) + 1)

In [6]:
model = Sequential([
    layers.Embedding(MAX_VOCAB, 128, input_length=MAX_LEN, mask_zero=True),
    layers.SpatialDropout1D(0.3),
    layers.LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3),
    layers.LSTM(64, return_sequences=True, dropout=0.3, recurrent_dropout=0.3),
    layers.LSTM(32, dropout=0.3, recurrent_dropout=0.3),
    layers.Dropout(0.3),
    layers.Dense(64, activation="tanh"),
    layers.Dropout(0.3),
    layers.Dense(len(classes), activation="softmax")
])

opt = optimizers.Adam(learning_rate=5e-4, clipnorm=1.0)
model.compile(loss="categorical_crossentropy", optimizer=opt, metrics=["accuracy"])

model.summary()

c:\Users\Laptop Dunyası\OneDrive\Masaüstü\sentiment_analysis\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:
cb = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)
]

history = model.fit(
    X_train_pad, y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=20,
    batch_size=128,
    callbacks=cb,
    verbose=1
)

Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 438s 2s/step - accuracy: 0.5950 - loss: 0.8754 - val_accuracy: 0.6830 - val_loss: 0.7709 - learning_rate: 5.0000e-04
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 235s 1s/step - accuracy: 0.7633 - loss: 0.6136 - val_accuracy: 0.7285 - val_loss: 0.6662 - learning_rate: 5.0000e-04
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 254s 1s/step - accuracy: 0.8153 - loss: 0.4963 - val_accuracy: 0.7582 - val_loss: 0.6213 - learning_rate: 5.0000e-04
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 193s 1s/step - accuracy: 0.8457 - loss: 0.4209 - val_accuracy: 0.7655 - val_loss: 0.6307 - learning_rate: 5.0000e-04
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 3044s 16s/step - accuracy: 0.8661 - loss: 0.3628 - val_accuracy: 0.7640 - val_loss: 0.6681 - learning_rate: 5.0000e-04
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 210s 1s/step - accuracy: 0.8873 - loss: 0.3121 - val_accuracy: 0.7663 - val_loss: 0.6873 - learning_rate: 2.5000e-04


In [ ]:
y_true = y_val_cat.argmax(axis=1)
y_pred = model.predict(X_val_pad).argmax(axis=1)
print(classification_report(y_true, y_pred, target_names=classes))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))

188/188 ━━━━━━━━━━━━━━━━━━━━ 21s 108ms/step
              precision    recall  f1-score   support

     negatif       0.72      0.76      0.74      2000
        notr       0.78      0.69      0.73      2000
     pozitif       0.77      0.82      0.80      2000

    accuracy                           0.76      6000
   macro avg       0.76      0.76      0.76      6000
weighted avg       0.76      0.76      0.76      6000

Macro F1: 0.757481024507975


In [11]:
def predict_text(text, model=model, tok=tok, classes=None, maxlen=MAX_LEN):
    if classes is None:
        classes = list(classes)
    seq  = tok.texts_to_sequences([text])
    pad  = pad_sequences(seq, maxlen=maxlen, padding="post", truncating="post")
    probs = model.predict(pad, verbose=0)[0]
    cls_idx = int(probs.argmax())
    return classes[cls_idx], probs


examples = [
    "Kötü",
    "Ürün gerçekten harika, çok memnun kaldım.",
    "Kargo çok geç geldi ve paket yırtıktı.",
    "Ürün fena değil, idare eder.",
    "Telefonun özellikleri iyi ama bataryası çabuk bitiyor.",
    "Ürün güzel ama fiyatına göre performansı düşük.",
    "Kargo biraz geç geldi ama satıcı çok yardımcı oldu.",
    "Sevdim ürünü, tekrar alırım.",
    "Tavsiye ederim.",
    "Berbat"
]

for ex in examples:
    label, probs = predict_text(ex, model=model, tok=tok, classes=classes, maxlen=100)
    print(f"Metin: {ex}")
    print(f"Tahmin: {label}, Olasılıklar: {probs}\n")


Metin: Kötü
Tahmin: negatif, Olasılıklar: [0.46804106 0.34610796 0.18585095]

Metin: Ürün gerçekten harika, çok memnun kaldım.
Tahmin: pozitif, Olasılıklar: [0.02044384 0.0877072  0.8918489 ]

Metin: Kargo çok geç geldi ve paket yırtıktı.
Tahmin: notr, Olasılıklar: [0.2667428  0.7110899  0.02216726]

Metin: Ürün fena değil, idare eder.
Tahmin: notr, Olasılıklar: [0.16486397 0.82365966 0.01147638]

Metin: Telefonun özellikleri iyi ama bataryası çabuk bitiyor.
Tahmin: notr, Olasılıklar: [0.18475302 0.78904665 0.02620037]

Metin: Ürün güzel ama fiyatına göre performansı düşük.
Tahmin: notr, Olasılıklar: [0.03050425 0.92350084 0.04599489]

Metin: Kargo biraz geç geldi ama satıcı çok yardımcı oldu.
Tahmin: notr, Olasılıklar: [0.11132371 0.8844435  0.00423281]

Metin: Sevdim ürünü, tekrar alırım.
Tahmin: pozitif, Olasılıklar: [0.11446209 0.31811163 0.5674263 ]

Metin: Tavsiye ederim.
Tahmin: pozitif, Olasılıklar: [0.22921482 0.3029195  0.46786562]

Metin: Berbat
Tahmin: negatif, Olasılıklar:

In [12]:
from pathlib import Path

current_dir = Path.cwd() / "Models"
save_path = current_dir / "stacked_lstm_model.keras"

model.save(save_path)
print(f"Stacked LSTM modeli kaydedildi: {save_path}")


Stacked LSTM modeli kaydedildi: c:\Users\Laptop Dunyası\OneDrive\Masaüstü\sentiment_analysis\Models\stacked_lstm_model.keras
